# Lab 2b — Import a pre-trained model (skip training)

Use this notebook **instead of notebook 2** when no GPU is available for training, or when there is no time to
wait for a training job. It registers a model exactly as notebook 2 does, so notebooks 3 (evaluation) and 4
(deployment) run unchanged.

By default it uses the **LoRA adapter shipped in this repo** (`pretrained/libero-r8/`, ~40 MB, trained on the
LIBERO sample with `recipes/openvla_lora.yaml`; see its README). Only the adapter is uploaded; the evaluation job
and the endpoint download the public base model `openvla/openvla-7b` from the Hugging Face Hub and merge the
adapter into it when they start.

You can instead point `PRETRAINED_S3_URI` at the output of a previous run (`.../output/model/`, holding
`hf_merged/` or `adapter/`). A prefix outside this account's bucket is copied in first.

Notebook 1 must have run in this account first (CPU only): notebook 3 evaluates on its validation split.


In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

In [ ]:
import time
from pathlib import Path

from tqdm.auto import tqdm

from artifacts import copy_prefix, register_model, split_s3

PRETRAINED_LOCAL_DIR = config.PRETRAINED_LOCAL_DIR   # adapter shipped in the repo (default)
PRETRAINED_S3_URI = config.PRETRAINED_S3_URI         # or e.g. "s3://<bucket>/openvla-lora/training-output/<job>/output/model/"
PRETRAINED_RUN_MODE = "full"                         # recorded in the model package

## Upload (or copy) the model and register it

In [ ]:
if PRETRAINED_S3_URI:
    source = PRETRAINED_S3_URI.rstrip("/") + "/"
    if source.startswith(s3_root.rstrip("/") + "/"):
        model_prefix = source
        print("already in this account's bucket: registering in place")
    else:
        model_prefix = f"{s3_root}/pretrained/{time.strftime('%Y%m%d-%H%M%S')}/"
        print(f"copied {copy_prefix(source, model_prefix, region=region, progress=tqdm)} objects to {model_prefix}")
    keys = {o["Key"].rsplit("/", 2)[-2] for o in s3.list_objects_v2(Bucket=split_s3(model_prefix)[0],
                                                                    Prefix=split_s3(model_prefix)[1])["Contents"]}
    model_s3_uri = model_prefix + ("hf_merged/" if "hf_merged" in keys else "adapter/")
else:
    source = f"repo:{PRETRAINED_LOCAL_DIR}"
    model_s3_uri = f"{s3_root}/pretrained/{Path(PRETRAINED_LOCAL_DIR).name}/"
    bucket, key = split_s3(model_s3_uri)
    files = [f for f in Path(PRETRAINED_LOCAL_DIR).iterdir() if f.is_file() and f.suffix in (".json", ".safetensors")]
    for f in files:
        s3.upload_file(str(f), bucket, key + f.name)
    print(f"uploaded {[f.name for f in files]} to {model_s3_uri}")

package = register_model(model_s3_uri, source=source, run_mode=PRETRAINED_RUN_MODE, region=region)
print(f"model package: {package['ModelPackageArn']}")

Continue with **3-evaluation.ipynb**.